human lung cancer

导入相关包

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)


In [2]:
repeat_times = 2
shard_inx = [0,1,2]
simple_path = [f'/home/cavin/jt/benchmark/data/crc/VisiumHD_P2_shard_{shard_inx}.h5ad' for shard_inx in shard_inx]
cell_emb_col = 'X_GeneFormer'
save_path = [f"/home/cavin/jt/benchmark/experiments/embedings/spatial_cluster_no_annotations/geneformer_human_CRC_shard_{shard}_emb.parquet" for shard in shard_inx]


In [3]:
adata = [sc.read_h5ad(path) for path in simple_path]
var = adata[0].var
adata = sc.concat(adata, axis=0)
adata.var = var
del var
adata

AnnData object with n_obs × n_vars = 500470 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
    var: 'ensembl_id', 'gene_name'
    obsm: 'spatial'

读取嵌入

In [4]:
loaded_emb_df = [pd.read_parquet(s) for s in save_path]
loaded_emb_df = pd.concat(loaded_emb_df)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
del loaded_emb_df
adata.obsm[cell_emb_col] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 500470 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
    var: 'ensembl_id', 'gene_name'
    obsm: 'spatial', 'X_GeneFormer'

In [5]:
res_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
random_seed=2026 + repeat_times * 200
key_added='leiden'
sc.pp.neighbors(adata, use_rep=cell_emb_col,random_state=random_seed)
adata


AnnData object with n_obs × n_vars = 500470 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
    var: 'ensembl_id', 'gene_name'
    uns: 'neighbors'
    obsm: 'spatial', 'X_GeneFormer'
    obsp: 'distances', 'connectivities'

In [6]:
def main(adata,random_seed,res_list,key_added):
    from tqdm import tqdm
    for used_res in tqdm(res_list):
        sc.tl.leiden(adata, random_state=random_seed, resolution=used_res,key_added=f"{key_added}_{used_res}")


In [7]:
main(adata,random_seed,res_list,key_added)

  0%|                                              | 0/10 [00:00<?, ?it/s]

/tmp/ipykernel_2407117/2761875232.py:4: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, random_state=random_seed, resolution=used_res,key_added=f"{key_added}_{used_res}")


 10%|███▋                                 | 1/10 [01:48<16:14, 108.30s/it]

 20%|███████▍                             | 2/10 [07:33<33:03, 247.94s/it]

 30%|███████████                          | 3/10 [12:55<32:51, 281.64s/it]

...

 70%|█████████████████████████▉           | 7/10 [28:36<12:26, 248.74s/it]

 80%|█████████████████████████████▌       | 8/10 [35:05<09:46, 293.41s/it]

 90%|█████████████████████████████████▎   | 9/10 [44:03<06:09, 369.90s/it]

100%|████████████████████████████████████| 10/10 [53:25<00:00, 429.33s/it]

100%|████████████████████████████████████| 10/10 [53:25<00:00, 320.57s/it]

In [8]:
for used_res in res_list:
    save_label_df_path = f"/home/cavin/jt/benchmark/experiments/results/labels_df/CRC/geneformer_human_CRC_labels_repeat_{repeat_times}_resolution_{used_res}.csv"
    adata.obs[f"{key_added}_{used_res}"].to_csv(save_label_df_path)